# EasyOCR Parameter Tuning (Vietnamese Sheet Layout)

This notebook builds a small, testable workflow to tune EasyOCR detection parameters.

Goal:
- Generate Vietnamese handwriting sheets with default sheet arrangement settings.
- Score detection quality against expected layout anchors derived from the same sheet rules.
- Export ranked parameter candidates and baseline-vs-best visual overlays.

## 1. Set Up Notebook Environment

Import core libraries, set deterministic seeds, and configure notebook display/logging.

In [1]:
from __future__ import annotations

import json
import logging
import random
import sys
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 120)

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
logger = logging.getLogger("easyocr-parameter-tuning")
logger.setLevel(logging.INFO)

print(f"Seed set to {SEED}")

Seed set to 42


## 2. Define Project Configuration

Create reusable config objects for paths, runtime flags, and sample/tuning parameters.

In [2]:
def discover_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "config" / "default.yaml").exists():
            return candidate
    raise FileNotFoundError("Could not locate project root from current directory")


PROJECT_ROOT = discover_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from api.dependencies import get_config
from api.services import build_sheet_options
from src.easy_ocr.easyocr import EASYOCR_DETECT_DEFAULTS, EasyOCRDetector
from src.utils.handwriting_sheet_generation import generate_handwriting_sheet


@dataclass(frozen=True)
class NotebookConfig:
    project_root: Path
    reports_dir: Path
    processed_dir: Path
    sample_count: int = 4
    tokens_per_sample: int = 72
    random_search_size: int = 24
    detect_tolerance_px: int = 80
    row_cluster_tol_ratio: float = 0.65
    dpi: int = 300
    output_prefix: str = "easyocr_tune_vi"
    use_gpu: bool = False


NOTEBOOK_CFG = NotebookConfig(
    project_root=PROJECT_ROOT,
    reports_dir=PROJECT_ROOT / "reports",
    processed_dir=PROJECT_ROOT / "data" / "processed" / "easyocr",
)

NOTEBOOK_CFG.reports_dir.mkdir(parents=True, exist_ok=True)
NOTEBOOK_CFG.processed_dir.mkdir(parents=True, exist_ok=True)

print(f"Project root: {NOTEBOOK_CFG.project_root}")
print(f"Reports dir: {NOTEBOOK_CFG.reports_dir}")
print(f"Processed dir: {NOTEBOOK_CFG.processed_dir}")

Project root: /home/tp_ubuntu/unsorted/handwriting-recognition
Reports dir: /home/tp_ubuntu/unsorted/handwriting-recognition/reports
Processed dir: /home/tp_ubuntu/unsorted/handwriting-recognition/data/processed/easyocr


## 3. Create Core Data Structures

Define sample payloads, candidate payloads, and helper utilities for layout references and detection metrics.

In [3]:
@dataclass
class SheetSample:
    sample_id: str
    image_path: Path
    tokens: list[str]
    expected_rows: int
    expected_boxes: int
    expected_anchors: list[tuple[float, float]]
    line_spacing_px: float
    sheet_defaults: dict[str, int]


@dataclass
class CandidateSpec:
    candidate_id: int
    candidate_name: str
    params: dict[str, Any]


def resolve_from_project(project_root: Path, raw_path: str) -> Path:
    candidate = Path(str(raw_path)).expanduser()
    if not candidate.is_absolute():
        candidate = project_root / candidate
    return candidate.resolve()


def select_first_font(fonts_dir: Path) -> Path:
    extensions = {".ttf", ".otf"}
    if not fonts_dir.exists() or not fonts_dir.is_dir():
        raise FileNotFoundError(f"Font directory is missing: {fonts_dir}")

    for font_path in sorted(fonts_dir.iterdir()):
        if font_path.is_file() and font_path.suffix.lower() in extensions:
            return font_path.resolve()

    raise FileNotFoundError(
        f"No .ttf/.otf font found in directory: {fonts_dir}"
    )


def build_token_variants(
    base_tokens: list[str],
    sample_count: int,
    tokens_per_sample: int,
) -> list[list[str]]:
    if not base_tokens:
        raise ValueError("vi_text produced no tokens")

    total = len(base_tokens)
    window = max(1, min(tokens_per_sample, total))
    variants: list[list[str]] = []

    for index in range(sample_count):
        start = (index * window) % total
        variant = [base_tokens[(start + offset) % total] for offset in range(window)]
        variants.append(variant)

    return variants


def pt_to_px(value_points: float, dpi: int = 300) -> float:
    return float(value_points) * float(dpi) / 72.0


def estimate_vi_layout_reference(
    tokens: list[str],
    defaults: dict[str, int],
    *,
    page_height_points: int,
    dpi: int,
) -> dict[str, Any]:
    margin_left = int(defaults["margin_left"])
    margin_right = int(defaults["margin_right"])
    margin_top = int(defaults["margin_top"])
    margin_bottom = int(defaults["margin_bottom"])
    divide_vertical = int(defaults["divide_vertical"])
    line_spacing = int(defaults["line_spacing"])
    word_spacing = int(defaults["word_spacing"])
    font_size = int(defaults["font_size"])

    x_pos = margin_left
    y_pos = margin_top
    left_column = True
    page_index = 0

    anchors: list[tuple[float, float]] = []
    row_y_centers_px: list[float] = []

    for token in tokens:
        right_limit = divide_vertical if left_column else margin_right
        x = x_pos

        row_y_center_pt = y_pos + (font_size * 0.35)
        row_y_center_px = pt_to_px(page_height_points - row_y_center_pt, dpi)
        row_y_centers_px.append(row_y_center_px)

        while True:
            x_center_pt = x + (font_size * 0.35)
            y_center_pt = y_pos + (font_size * 0.35)
            anchors.append(
                (
                    pt_to_px(x_center_pt, dpi),
                    pt_to_px(page_height_points - y_center_pt, dpi),
                )
            )

            x += word_spacing
            if x > (right_limit - 10):
                break

        y_pos -= line_spacing
        if y_pos < margin_bottom:
            if left_column:
                x_pos = divide_vertical + 10
                y_pos = margin_top
                left_column = False
            else:
                page_index += 1
                x_pos = margin_left
                y_pos = margin_top
                left_column = True

    return {
        "expected_rows": len(tokens),
        "expected_boxes": len(anchors),
        "anchors": anchors,
        "line_spacing_px": pt_to_px(line_spacing, dpi),
        "page_count": page_index + 1,
        "row_y_centers_px": row_y_centers_px,
    }


def normalize_horizontal_boxes(detect_output: dict[str, Any]) -> list[list[int]]:
    if not isinstance(detect_output, dict):
        return []

    raw_boxes = detect_output.get("horizontal_list", [])
    if not isinstance(raw_boxes, list):
        return []

    normalized: list[list[int]] = []
    for box in raw_boxes:
        if not isinstance(box, (list, tuple)) or len(box) < 4:
            continue

        try:
            x_min = int(round(float(box[0])))
            x_max = int(round(float(box[1])))
            y_min = int(round(float(box[2])))
            y_max = int(round(float(box[3])))
        except (TypeError, ValueError):
            continue

        if x_max <= x_min or y_max <= y_min:
            continue

        normalized.append([x_min, x_max, y_min, y_max])

    return normalized


def box_center(box: list[int]) -> tuple[float, float]:
    x_center = (float(box[0]) + float(box[1])) / 2.0
    y_center = (float(box[2]) + float(box[3])) / 2.0
    return x_center, y_center


def cluster_rows_by_y(
    centers: list[tuple[float, float]],
    tolerance_px: float,
) -> list[list[tuple[float, float]]]:
    if not centers:
        return []

    ordered = sorted(centers, key=lambda item: item[1])
    rows: list[list[tuple[float, float]]] = [[ordered[0]]]

    for center in ordered[1:]:
        if abs(center[1] - rows[-1][-1][1]) <= tolerance_px:
            rows[-1].append(center)
        else:
            rows.append([center])

    return rows


def greedy_anchor_match(
    detected_centers: list[tuple[float, float]],
    expected_anchors: list[tuple[float, float]],
    tolerance_px: float,
) -> int:
    if not detected_centers or not expected_anchors:
        return 0

    unused = set(range(len(detected_centers)))
    matched = 0

    for anchor_x, anchor_y in expected_anchors:
        best_idx: int | None = None
        best_distance = float("inf")

        for index in unused:
            detect_x, detect_y = detected_centers[index]
            distance = ((detect_x - anchor_x) ** 2 + (detect_y - anchor_y) ** 2) ** 0.5
            if distance <= tolerance_px and distance < best_distance:
                best_distance = distance
                best_idx = index

        if best_idx is not None:
            unused.remove(best_idx)
            matched += 1

    return matched


def compute_detection_metrics(
    boxes: list[list[int]],
    sample: SheetSample,
    *,
    detect_tolerance_px: float,
    row_cluster_tol_ratio: float,
) -> dict[str, float | int]:
    expected_total = max(1, int(sample.expected_boxes))
    detected_total = len(boxes)

    detected_centers = [box_center(box) for box in boxes]
    matched = greedy_anchor_match(
        detected_centers,
        sample.expected_anchors,
        tolerance_px=detect_tolerance_px,
    )

    coverage = float(matched) / float(expected_total)
    extra_penalty = float(max(0, detected_total - sample.expected_boxes)) / float(expected_total)
    missing_penalty = float(max(0, sample.expected_boxes - detected_total)) / float(expected_total)

    row_tol_px = max(4.0, sample.line_spacing_px * float(row_cluster_tol_ratio))
    detected_rows = len(cluster_rows_by_y(detected_centers, row_tol_px))
    row_error = float(abs(detected_rows - sample.expected_rows)) / float(max(1, sample.expected_rows))
    row_alignment = max(0.0, 1.0 - row_error)

    composite = (
        (0.60 * coverage)
        + (0.30 * row_alignment)
        - (0.07 * extra_penalty)
        - (0.03 * missing_penalty)
    )

    return {
        "expected_boxes": sample.expected_boxes,
        "detected_boxes": detected_total,
        "expected_rows": sample.expected_rows,
        "detected_rows": detected_rows,
        "matched_anchors": matched,
        "coverage": coverage,
        "row_alignment": row_alignment,
        "extra_penalty": extra_penalty,
        "missing_penalty": missing_penalty,
        "composite": composite,
    }

## 4. Implement Main Processing Function

Build the end-to-end workflow for dataset generation, parameter candidate creation, candidate evaluation, ranking, and artifact export.

In [4]:
DETECT_KEYS: list[str] = [
    "min_size",
    "text_threshold",
    "low_text",
    "link_threshold",
    "canvas_size",
    "mag_ratio",
    "slope_ths",
    "ycenter_ths",
    "height_ths",
    "width_ths",
    "add_margin",
    "optimal_num_chars",
]


def clamp(value: float, low: float, high: float) -> float:
    return max(low, min(high, value))


def load_runtime_defaults() -> tuple[Any, dict[str, int], dict[str, Any]]:
    app_config = get_config()
    options = build_sheet_options(app_config)
    sheet_defaults = dict(options["defaults"])

    inference = getattr(app_config, "inference", None)
    detect_cfg = getattr(inference, "easyocr_detect", None)

    base_params: dict[str, Any] = dict(EASYOCR_DETECT_DEFAULTS)
    if detect_cfg is not None:
        for key in DETECT_KEYS:
            if hasattr(detect_cfg, key):
                value = getattr(detect_cfg, key)
                base_params[key] = value

    # Keep integers as integers for stable detect() kwargs.
    base_params["min_size"] = int(base_params.get("min_size", 3))
    base_params["canvas_size"] = int(base_params.get("canvas_size", 2560))

    optional_chars = base_params.get("optimal_num_chars")
    if optional_chars is not None:
        base_params["optimal_num_chars"] = int(optional_chars)

    return app_config, sheet_defaults, base_params


def build_candidate_space(
    base_params: dict[str, Any],
    *,
    random_search_size: int,
    seed: int,
) -> list[CandidateSpec]:
    rng = random.Random(seed)
    candidates: list[CandidateSpec] = []
    seen: set[str] = set()

    def normalize_candidate(raw_params: dict[str, Any]) -> dict[str, Any]:
        normalized = dict(base_params)
        normalized.update(raw_params)

        normalized["min_size"] = int(clamp(float(normalized["min_size"]), 1, 64))
        normalized["canvas_size"] = int(clamp(float(normalized["canvas_size"]), 1024, 4096))

        for key in [
            "text_threshold",
            "low_text",
            "link_threshold",
            "mag_ratio",
            "slope_ths",
            "ycenter_ths",
            "height_ths",
            "width_ths",
            "add_margin",
        ]:
            if key == "mag_ratio":
                normalized[key] = float(clamp(float(normalized[key]), 0.5, 6.0))
            elif key == "add_margin":
                normalized[key] = float(clamp(float(normalized[key]), 0.0, 0.3))
            else:
                normalized[key] = float(clamp(float(normalized[key]), 0.0, 1.0))

        optional_chars = normalized.get("optimal_num_chars")
        normalized["optimal_num_chars"] = None if optional_chars is None else int(optional_chars)

        return normalized

    def push(name: str, raw_params: dict[str, Any]) -> None:
        normalized = normalize_candidate(raw_params)
        signature = json.dumps(normalized, sort_keys=True)
        if signature in seen:
            return
        seen.add(signature)

        candidates.append(
            CandidateSpec(
                candidate_id=len(candidates),
                candidate_name=name,
                params=normalized,
            )
        )

    push("baseline_config", base_params)
    push("legacy_code_defaults", dict(EASYOCR_DETECT_DEFAULTS))

    for index in range(random_search_size):
        proposal = dict(base_params)
        proposal["min_size"] = int(round(float(base_params["min_size"]) + rng.randint(-2, 4)))
        proposal["text_threshold"] = float(base_params["text_threshold"]) + rng.uniform(-0.20, 0.20)
        proposal["low_text"] = float(base_params["low_text"]) * rng.uniform(0.4, 2.0)
        proposal["link_threshold"] = float(base_params["link_threshold"]) * rng.uniform(0.5, 2.2)
        proposal["canvas_size"] = int(float(base_params["canvas_size"]) + rng.choice([-512, -256, 0, 256, 512]))
        proposal["mag_ratio"] = float(base_params["mag_ratio"]) * rng.uniform(0.7, 1.6)
        proposal["slope_ths"] = float(base_params["slope_ths"]) * rng.uniform(0.7, 1.5)
        proposal["ycenter_ths"] = float(base_params["ycenter_ths"]) * rng.uniform(0.6, 1.6)
        proposal["height_ths"] = float(base_params["height_ths"]) * rng.uniform(0.6, 1.6)
        proposal["width_ths"] = float(base_params["width_ths"]) * rng.uniform(0.6, 1.6)
        proposal["add_margin"] = float(base_params["add_margin"]) * rng.uniform(0.6, 1.6)
        proposal["optimal_num_chars"] = None

        push(f"random_{index:02d}", proposal)

    return candidates


def generate_vi_sheet_samples(
    cfg: NotebookConfig,
    *,
    app_config: Any,
    sheet_defaults: dict[str, int],
    font_path: Path,
) -> list[SheetSample]:
    vi_text = str(getattr(app_config.sheet, "vi_text", "")).strip()
    tokens = vi_text.split()
    variants = build_token_variants(
        tokens,
        sample_count=cfg.sample_count,
        tokens_per_sample=cfg.tokens_per_sample,
    )

    page_height_points = int(getattr(app_config.sheet, "height", 841))

    samples: list[SheetSample] = []
    for index, token_variant in enumerate(variants):
        sample_id = f"sample_{index:02d}"
        output_basename = f"{cfg.output_prefix}_{sample_id}"
        custom_text = " ".join(token_variant)

        generation = generate_handwriting_sheet(
            app_config,
            font_path=str(font_path),
            language="vi",
            custom_text=custom_text,
            overrides=sheet_defaults,
            output_basename=output_basename,
        )

        image_path = Path(str(generation.get("preview_png_path", ""))).resolve()
        if not image_path.exists():
            raise FileNotFoundError(f"Preview image was not generated: {image_path}")

        reference = estimate_vi_layout_reference(
            token_variant,
            sheet_defaults,
            page_height_points=page_height_points,
            dpi=cfg.dpi,
        )

        samples.append(
            SheetSample(
                sample_id=sample_id,
                image_path=image_path,
                tokens=token_variant,
                expected_rows=int(reference["expected_rows"]),
                expected_boxes=int(reference["expected_boxes"]),
                expected_anchors=list(reference["anchors"]),
                line_spacing_px=float(reference["line_spacing_px"]),
                sheet_defaults=dict(sheet_defaults),
            )
        )

    return samples


def evaluate_candidates(
    cfg: NotebookConfig,
    detector: EasyOCRDetector,
    candidates: list[CandidateSpec],
    samples: list[SheetSample],
) -> tuple[pd.DataFrame, pd.DataFrame]:
    detail_rows: list[dict[str, Any]] = []

    for candidate in candidates:
        for sample in samples:
            detect_output = detector.detect_char_boxes(
                str(sample.image_path),
                **candidate.params,
            )
            boxes = normalize_horizontal_boxes(detect_output)
            metrics = compute_detection_metrics(
                boxes,
                sample,
                detect_tolerance_px=cfg.detect_tolerance_px,
                row_cluster_tol_ratio=cfg.row_cluster_tol_ratio,
            )

            detail_rows.append(
                {
                    "candidate_id": candidate.candidate_id,
                    "candidate_name": candidate.candidate_name,
                    "sample_id": sample.sample_id,
                    **candidate.params,
                    **metrics,
                }
            )

    detail_df = pd.DataFrame(detail_rows)

    summary_df = (
        detail_df.groupby(["candidate_id", "candidate_name"], as_index=False)
        .agg(
            mean_composite=("composite", "mean"),
            mean_coverage=("coverage", "mean"),
            mean_row_alignment=("row_alignment", "mean"),
            mean_extra_penalty=("extra_penalty", "mean"),
            mean_missing_penalty=("missing_penalty", "mean"),
            std_composite=("composite", "std"),
            samples=("sample_id", "count"),
        )
        .fillna({"std_composite": 0.0})
        .sort_values(
            by=["mean_composite", "mean_coverage", "mean_row_alignment"],
            ascending=False,
        )
        .reset_index(drop=True)
    )

    return detail_df, summary_df


def draw_overlay(
    image_path: Path,
    boxes: list[list[int]],
    anchors: list[tuple[float, float]],
    out_path: Path,
) -> None:
    image = Image.open(image_path).convert("RGB")
    drawer = ImageDraw.Draw(image)

    for box in boxes:
        drawer.rectangle(
            [int(box[0]), int(box[2]), int(box[1]), int(box[3])],
            outline=(20, 220, 40),
            width=2,
        )

    step = max(1, len(anchors) // 800)
    for x_coord, y_coord in anchors[::step]:
        x_pixel = int(round(x_coord))
        y_pixel = int(round(y_coord))
        radius = 2
        drawer.ellipse(
            [x_pixel - radius, y_pixel - radius, x_pixel + radius, y_pixel + radius],
            fill=(255, 140, 0),
            outline=(255, 140, 0),
        )

    out_path.parent.mkdir(parents=True, exist_ok=True)
    image.save(out_path)


def run_tuning_workflow(cfg: NotebookConfig) -> dict[str, Any]:
    app_config, sheet_defaults, base_params = load_runtime_defaults()

    fonts_dir = resolve_from_project(
        cfg.project_root,
        str(getattr(app_config.sheet, "fonts_dir", "data/reference/fonts")),
    )
    font_path = select_first_font(fonts_dir)

    logger.info("Selected font: %s", font_path.name)
    logger.info("Building synthetic Vietnamese sheet samples")
    samples = generate_vi_sheet_samples(
        cfg,
        app_config=app_config,
        sheet_defaults=sheet_defaults,
        font_path=font_path,
    )

    logger.info("Generated %d samples", len(samples))
    candidates = build_candidate_space(
        base_params,
        random_search_size=cfg.random_search_size,
        seed=SEED,
    )
    logger.info("Evaluating %d candidate parameter sets", len(candidates))

    detector = EasyOCRDetector(languages=["vi"], gpu=cfg.use_gpu)
    detail_df, summary_df = evaluate_candidates(
        cfg,
        detector,
        candidates,
        samples,
    )

    if summary_df.empty:
        raise RuntimeError("No candidate evaluation results were produced")

    candidate_lookup = {candidate.candidate_id: candidate for candidate in candidates}

    best_row = summary_df.iloc[0]
    best_candidate = candidate_lookup[int(best_row["candidate_id"])]

    baseline_candidates = [
        candidate for candidate in candidates if candidate.candidate_name == "baseline_config"
    ]
    baseline_candidate = baseline_candidates[0] if baseline_candidates else best_candidate

    first_sample = samples[0]
    overlay_dir = cfg.processed_dir / "tuning_overlays"

    for candidate, name in [
        (baseline_candidate, "baseline"),
        (best_candidate, "best"),
    ]:
        detect_output = detector.detect_char_boxes(
            str(first_sample.image_path),
            **candidate.params,
        )
        boxes = normalize_horizontal_boxes(detect_output)
        draw_overlay(
            first_sample.image_path,
            boxes,
            first_sample.expected_anchors,
            overlay_dir / f"{cfg.output_prefix}_{name}_overlay.png",
        )

    ranking_path = cfg.reports_dir / "easyocr_parameter_tuning_vietnamese.json"
    payload = {
        "seed": SEED,
        "font": font_path.name,
        "sample_count": len(samples),
        "tokens_per_sample": cfg.tokens_per_sample,
        "base_params": base_params,
        "best": {
            "candidate_id": int(best_candidate.candidate_id),
            "candidate_name": str(best_candidate.candidate_name),
            "params": best_candidate.params,
            "mean_composite": float(best_row["mean_composite"]),
            "mean_coverage": float(best_row["mean_coverage"]),
            "mean_row_alignment": float(best_row["mean_row_alignment"]),
        },
        "ranked": summary_df.to_dict(orient="records"),
    }
    ranking_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")

    return {
        "sheet_defaults": sheet_defaults,
        "base_params": base_params,
        "samples": samples,
        "candidates": candidates,
        "detail_df": detail_df,
        "summary_df": summary_df,
        "ranking_path": ranking_path,
        "overlay_dir": overlay_dir,
        "best_candidate": best_candidate,
        "baseline_candidate": baseline_candidate,
    }

In [6]:
def select_first_font(fonts_dir: Path) -> Path:
    """Prefer configured fonts_dir; fallback to other local font files."""
    extensions = {".ttf", ".otf"}

    search_roots = [
        fonts_dir,
        fonts_dir.parent,
        NOTEBOOK_CFG.project_root / "data" / "reference",
    ]

    seen: set[Path] = set()
    for root in search_roots:
        root = root.resolve()
        if root in seen:
            continue
        seen.add(root)

        if not root.exists() or not root.is_dir():
            continue

        for font_path in sorted(root.rglob("*")):
            if font_path.is_file() and font_path.suffix.lower() in extensions:
                logger.warning(
                    "Using fallback font path: %s",
                    font_path,
                )
                return font_path.resolve()

    raise FileNotFoundError(
        "No .ttf/.otf font found under configured or fallback paths"
    )

## 5. Run a Minimal End-to-End Example

Execute one full tuning pass and inspect top-ranked candidates.

In [7]:
workflow = run_tuning_workflow(NOTEBOOK_CFG)

summary_df = workflow["summary_df"]
detail_df = workflow["detail_df"]

print(f"Ranking JSON: {workflow['ranking_path']}")
print(f"Overlay directory: {workflow['overlay_dir']}")
print(f"Candidates evaluated: {len(workflow['candidates'])}")
print(f"Samples evaluated: {len(workflow['samples'])}")

summary_df.head(10)

WARNING Using fallback font path: /home/tp_ubuntu/unsorted/handwriting-recognition/data/reference/arialbd.ttf
INFO Selected font: arialbd.ttf
INFO Building synthetic Vietnamese sheet samples
INFO Generated 4 samples
INFO Evaluating 26 candidate parameter sets
WARNING Using CPU. Note: This module is much faster with a GPU.


Ranking JSON: /home/tp_ubuntu/unsorted/handwriting-recognition/reports/easyocr_parameter_tuning_vietnamese.json
Overlay directory: /home/tp_ubuntu/unsorted/handwriting-recognition/data/processed/easyocr/tuning_overlays
Candidates evaluated: 26
Samples evaluated: 4


,candidate_id,candidate_name,mean_composite,mean_coverage,mean_row_alignment,mean_extra_penalty,mean_missing_penalty,std_composite,samples
0,1,legacy_code_defaults,0.171807,0.083887,0.496528,0.0,0.916113,0.013283,4
1,20,random_18,0.092674,0.035991,0.333333,0.0,0.964009,0.025100,4
2,22,random_20,0.092330,0.037099,0.329861,0.0,0.962901,0.034628,4
3,16,random_14,0.089719,0.034607,0.326389,0.0,0.965393,0.021565,4
4,8,random_06,0.088857,0.036545,0.319444,0.0,0.963455,0.034291,4
5,2,random_00,0.083643,0.034884,0.305556,0.0,0.965116,0.009397,4
6,5,random_03,0.082253,0.034330,0.302083,0.0,0.965670,0.024232,4
7,7,random_05,0.082083,0.035714,0.298611,0.0,0.964286,0.019230,4
8,23,random_21,0.072340,0.028516,0.281250,0.0,0.971484,0.015085,4
9,13,random_11,0.066434,0.027409,0.263889,0.0,0.972591,0.011097,4


## 6. Add Basic Validation Checks

Use assertions to verify key output shape, value bounds, and artifact generation.

In [8]:
if "workflow" not in globals():
    raise RuntimeError("Run the end-to-end example cell first.")

assert not summary_df.empty, "summary_df should not be empty"
assert not detail_df.empty, "detail_df should not be empty"
assert summary_df["mean_coverage"].between(0.0, 1.0).all(), "coverage must be within [0, 1]"
assert summary_df["mean_row_alignment"].between(0.0, 1.0).all(), "row alignment must be within [0, 1]"
assert summary_df["samples"].min() >= 1, "each candidate must evaluate at least one sample"

best_candidate_name = workflow["best_candidate"].candidate_name
assert best_candidate_name in set(summary_df["candidate_name"]), "best candidate should be listed in summary"
assert workflow["ranking_path"].exists(), "ranking JSON should exist"

print("Validation checks passed.")

Validation checks passed.


## 7. Save Outputs and Next-Step Hooks

Persist best-parameter artifacts and provide TODO hooks for runtime integration.

In [9]:
if "workflow" not in globals():
    raise RuntimeError("Run the end-to-end example cell first.")

best_candidate = workflow["best_candidate"]
best_export_path = NOTEBOOK_CFG.reports_dir / "easyocr_best_params_vietnamese.json"

best_payload = {
    "candidate_id": best_candidate.candidate_id,
    "candidate_name": best_candidate.candidate_name,
    "params": best_candidate.params,
    "source_ranking_file": str(workflow["ranking_path"]),
}
best_export_path.write_text(json.dumps(best_payload, indent=2), encoding="utf-8")

print(f"Saved best candidate params: {best_export_path}")
print("TODO: Apply these params into runtime scoring by passing detect kwargs from config in api/services.py")


def todo_apply_best_params_to_runtime() -> None:
    raise NotImplementedError(
        "TODO: Wire config.inference.easyocr_detect into detector.detect_char_boxes(...) in score_handwriting_sheet()."
    )

Saved best candidate params: /home/tp_ubuntu/unsorted/handwriting-recognition/reports/easyocr_best_params_vietnamese.json
TODO: Apply these params into runtime scoring by passing detect kwargs from config in api/services.py
